In [ ]:

import gym
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
#import tensorflow_probability as tfp
import random
import warnings
warnings.filterwarnings('ignore')
import math
print(gym.__version__)
import time 
from collections import deque
import warnings
warnings.filterwarnings('ignore')
from contextlib import redirect_stdout, redirect_stderr
import sys
import contextlib
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.losses import mean_squared_error, Huber


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

!nvidia-smi


In [ ]:
# Create the Cart-Pole environment
env = gym.make('CartPole-v0')
state_size = env.observation_space.shape[0]
num_actions = env.action_space.n
input_shape = (state_size,)

In [ ]:





'''

def combined_loss(y_true, y_pred):
    mse_loss = mean_squared_error(y_true, y_pred)
    huber_loss = Huber()(y_true, y_pred)
    return 0.5 * mse_loss + 0.5 * huber_loss  # Adjust the weights as needed

# Define the Deep Q-Network (DQN) model
def build_dqn_model(input_shape, num_actions):
    model = keras.Sequential([
        keras.layers.Dense(64, input_shape=input_shape, activation='relu', kernel_initializer='he_normal'),
        keras.layers.Dense(64, activation='relu'),
        keras.layers.Dense(num_actions, activation='linear')
    ])
    model.compile(optimizer='adam', loss=combined_loss)
    return model
'''
# Define the replay buffer
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def add(self, experience):
        self.buffer.append(experience)

    def sample(self, batch_size):
        return random.sample(self.buffer, batch_size)

In [ ]:


# Hyperparameters
epsilon = 0.95  # Exploration rate
epsilon_min = 0.01
epsilon_decay = 0.99
gamma = 0.95  # Discount factor
lr = 0.001
memory_capacity = 100000
batch_size = 256
target_update_frequency = 10   # Update the target network every n episodes
taw = 0.001

In [ ]:

dqn_model = tf.keras.Sequential()
dqn_model.add(tf.keras.Input(shape=input_shape))
dqn_model.add(tf.keras.layers.Dense(128, activation='linear',  kernel_initializer='he_normal'))
dqn_model.add(tf.keras.layers.Dense(128, activation='linear'))
dqn_model.add(tf.keras.layers.Dense(2, activation='linear'))



target_model = tf.keras.Sequential()
target_model.add(tf.keras.Input(shape=input_shape))
target_model.add(tf.keras.layers.Dense(128, activation='linear',  kernel_initializer='he_normal'))
target_model.add(tf.keras.layers.Dense(128, activation='linear'))
target_model.add(tf.keras.layers.Dense(2, activation='linear'))

In [ ]:


# Create DQN model and target model
#dqn_model = build_dqn_model((state_size,), num_actions)
#target_model = build_dqn_model((state_size,), num_actions)
target_model.set_weights(dqn_model.get_weights())
optimizer = keras.optimizers.Adam(learning_rate=lr )


# Create the replay buffer
replay_buffer = ReplayBuffer(memory_capacity)

# Training parameters
episodes = 700
episodes_rewards = []

In [ ]:


for episode in range(400, 2000):
    state = env.reset()[0]
    state = np.reshape(state, [1, state_size])
    state = tf.convert_to_tensor(state, dtype=tf.float32)
    total_reward = 0
    done = False
    terminated=False
    truncated =False
    mse_loss=0
    with tf.GradientTape(persistent=True) as tape:
        while (not terminated) and (not truncated):
            # Choose an action (epsilon-greedy policy)
            # For a probability of epislon, choose a random action 
            if np.random.rand() <= epsilon:
                action = env.action_space.sample()
            else:
            # For a probability of 1- epsilon choose the best action. 
                 with open('output.log', 'w') as log_file:
                    with contextlib.redirect_stdout(log_file), contextlib.redirect_stderr(log_file):
                        action = np.argmax(dqn_model(state))

            next_state, reward, terminated, truncated, info = env.step(action)
            next_state = np.reshape(next_state, [1, state_size])
            next_state = tf.convert_to_tensor(next_state, dtype=tf.float32)
            action = tf.convert_to_tensor(action, dtype=tf.int32)
            total_reward += reward
            reward = tf.convert_to_tensor(reward, dtype=tf.float32)
            # Store the experience in the replay buffer
            replay_buffer.add((state, action, reward, next_state, done))

            state = next_state
            if terminated or truncated:
                done = True

        # Update the DQN model using experience replay

        if len(replay_buffer.buffer) >= batch_size:
            batch = replay_buffer.sample(batch_size)
            for state, action, reward, next_state, done in batch:
                if not done:
                    with open('output.log', 'w') as log_file:
                        with contextlib.redirect_stdout(log_file), contextlib.redirect_stderr(log_file):
                            target = reward + gamma * np.amax(target_model(next_state)[0])
                else:
                    target = reward
                with open('output.log', 'w') as log_file:
                    with contextlib.redirect_stdout(log_file), contextlib.redirect_stderr(log_file):
                        target_f = dqn_model(state)
                target_f = tf.convert_to_tensor(target_f, dtype=tf.float32)     
                target = tf.convert_to_tensor(target, dtype=tf.float32)
                #target_f[0][action] = target
                #dqn_model.fit(state, target_f, epochs=1, verbose=0)
                #mse_loss = tf.keras.losses.mean_squared_error(Vt_eps, tensor_rewards_normalized)
                mse_loss = mse_loss + (target_f[0][action]-target)**2
    if len(replay_buffer.buffer) >= batch_size:         
        loss = tf.convert_to_tensor(mse_loss)
        grads = tape.gradient(loss, dqn_model.trainable_variables)
        optimizer.apply_gradients(zip(grads, dqn_model.trainable_variables))

    episodes_rewards.append(total_reward)
    # Decay exploration rate
    if epsilon > epsilon_min:
        epsilon *= epsilon_decay

    # Update the target network
    if episode % target_update_frequency == 0:
        #target_model.set_weights(taw*target_model.get_weights() + (1- taw)* dqn_model.get_weights())
        dqn_weights = dqn_model.get_weights()
        target_weights = target_model.get_weights()
        updated_weights = [taw * target + (1 - taw) * dqn for target, dqn in zip(target_weights, dqn_weights)]
        target_model.set_weights(updated_weights)


    # Print, plot and save models and data

    if episode<500:
        print(f"Episode: {episode}, Reward: {total_reward}, Epsilon: {epsilon}")

    if len(episodes_rewards) >= 100 and episode%100==0 :

        print(f"Episode: {episode}, Reward: {total_reward}")
        avg_reward = np.mean(episodes_rewards[-100:])
        print(f"Average Reward (Last 100 Episodes): {avg_reward}")

    #print(f"Episode: {episode + 1}, Total Reward: {total_reward}, Epsilon: {epsilon}")

    if episode%100==0 and episode!=0:
        dqn_model.save('dqn_model_'+str(episode))  # This will create a 'my_model' directory with model parameters
        target_model.save('target_model_'+str(episode))  # This will create a 'my_model' directory with model parameters
        checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
        filepath="model_checkpoint.h5",
        monitor="val_loss",
        save_best_only=True,
        save_weights_only=False,
        verbose=1, )


        warnings.filterwarnings('ignore')

        episode_numbers = list(range(1, len(episodes_rewards) + 1))
        #moving_avg = np.convolve(episodes_rewards, np.ones(100) / 100, mode='valid')
        plt.plot(episode_numbers, episodes_rewards, label="Episode Reward")
        #plt.plot(episode_numbers[-len(moving_avg):], moving_avg, label="Moving Average (100 episodes)")
        plt.xlabel("Episode")
        plt.ylabel("Reward")
        #plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
        #plt.show()
        plt.savefig('episode_rewards_'+ str(episode)+'.png')
        np_eps_rewards = np.array(episodes_rewards)
        np.save('np_eps_rewards_'+str(episode)+'.npy', np_eps_rewards)
        
# Save the trained model
dqn_model.save('cartpole_dqn_model.h5')
